# Activity 8: LSTM, a neural network that reads the monthly data

**Tuesday · Block 3** · Kernel: `nwcst` · About 25–30 minutes

The hands-on part of the LSTM lecture. Blocks 1 and 2 used one row per quarter, with each
predictor averaged over the quarter's three months. The LSTM instead reads the **monthly**
series directly: for every quarter it looks at a window of the last `n_timesteps` months and
learns which patterns in that window predict GDP growth.

You will:

1. Look at one training window, the object the network actually sees
2. Count the weights the network has to learn, against the number of windows it can learn from
3. Fit an LSTM and score it on the same 12 test quarters as Blocks 1–2
4. Move `train_episodes` and `n_timesteps` and read what 38 windows can and cannot teach a network
5. See what bagging (`n_models`) does
6. Measure how much one quarter (2020Q2) weighs in the network's level
7. Use the LSTM's strength: predict a quarter as it looked while the quarter was under way, when
   the latest months are still missing for several series (the ragged edge), and nowcast 2026Q2 and 2026Q3 with today's data

Everything runs on the CPU in about a minute. Cells marked **YOUR TURN** are yours.

## Preparation

The data were prepared exactly as in Activity 5 (build) and the nowcast tool: seasonally
adjusted at each series' own frequency, then turned into month-on-month growth rates (quarter
on quarter for GDP). Money and credit stay nominal, as in Blocks 1 and 2.

| File | What it holds |
|---|---|
| `blocks_monthly.csv` | The same 15 predictors, monthly, 2013-01 to 2026-10. `RGDP0000` appears only on quarter-end months (March, June, September, December) |
| `blocks_quarterly.csv` | Used here only for the train/test labels, so the test quarters match Blocks 1–2 |
| `blocks_meta.csv` | Short labels for charts |

The end of the monthly file is ragged: Brent and night lights already have October 2026, while
the credit series stop in July and cement sales in June. That is the normal state of a
nowcaster's data, and the LSTM is built to handle it.

Two settings in the cell below matter:

- **CPU only.** `nowcast_lstm` uses a GPU when it finds one; we force the CPU so everyone gets
  the same numbers.
- **Seeds.** Each of the `n_models` networks starts from random weights. Passing a fixed list
  of seeds makes the run reproducible.

Run as is.

In [ ]:
import os
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from nowcast_lstm.LSTM import LSTM

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

np.random.seed(42)
torch.manual_seed(42)

PATH_DATA = os.path.join("..", "data")
PATH_OUTPUT = os.path.join("..", "output")
os.makedirs(PATH_OUTPUT, exist_ok=True)
RESULTS_FILE = os.path.join(PATH_OUTPUT, "workshop_results.csv")
LANGUAGE = "Python"

target_variable = "RGDP0000"

monthly = pd.read_csv(os.path.join(PATH_DATA, "blocks_monthly.csv"), parse_dates=["date"])
quarterly = pd.read_csv(os.path.join(PATH_DATA, "blocks_quarterly.csv"), parse_dates=["date"])
meta = pd.read_csv(os.path.join(PATH_DATA, "blocks_meta.csv"))

predictors = [c for c in monthly.columns if c not in ("date", target_variable)]
LABELS = meta.set_index("ticker")["label"].to_dict()
labels = [LABELS.get(c, c) for c in predictors]

# nowcast_lstm wants: a datetime `date` column, the predictors, and the target LAST
monthly = monthly[["date"] + predictors + [target_variable]]

# Same split as Blocks 1-2
train_dates = quarterly.loc[quarterly["sample"] == "train", "date"]
test_dates = quarterly.loc[quarterly["sample"] == "test", "date"]
train_end = train_dates.max()          # 2023-03-01: last quarter of the train sample
test_end = test_dates.max()            # 2026-03-01: last published GDP quarter

y_obs = quarterly.set_index("date").loc[test_dates, target_variable]
y_train = quarterly.set_index("date").loc[train_dates, target_variable]

print(f"Monthly data : {monthly.date.min():%Y-%m} to {monthly.date.max():%Y-%m} ({len(monthly)} months)")
print(f"Train window : up to {train_end:%Y-%m} ({len(train_dates)} quarters of GDP)")
print(f"Test quarters: {test_dates.min():%Y-%m} to {test_end:%Y-%m} ({len(test_dates)})")

The helpers are the same as in Blocks 1 and 2 (`rmse`, `comparison_table`, `save_result`,
`results_table`, `plot_observed_predicted`), plus `benchmark_rmse`: the test error of predicting
the train-sample mean for every quarter, the line every model has to beat. Run and move on.

In [ ]:
RESULT_COLS = ["block", "activity", "language", "model", "rmse_test", "notes"]


def rmse(observed, predicted):
    """Root mean squared error."""
    return float(np.sqrt(np.mean((np.asarray(observed) - np.asarray(predicted)) ** 2)))


def comparison_table(predicted, name="Predicted"):
    """Observed vs predicted on the test sample, with the error in percentage points."""
    t = pd.DataFrame({"Observed": y_obs.values, name: np.asarray(predicted)}, index=y_obs.index)
    t["Error (pp)"] = t[name] - t["Observed"]
    return t.round(3)


def save_result(block, activity, model, rmse_value, notes=""):
    """Add or replace one row of the workshop results table (key: activity, language, model).
    Rows written by the R activities are kept untouched."""
    row = pd.DataFrame([{
        "block": int(block), "activity": activity, "language": LANGUAGE,
        "model": model, "rmse_test": round(float(rmse_value), 4), "notes": notes,
    }])
    if os.path.exists(RESULTS_FILE) and os.path.getsize(RESULTS_FILE) > 0:
        prev = pd.read_csv(RESULTS_FILE)
        same = ((prev["activity"] == activity) & (prev["language"] == LANGUAGE)
                & (prev["model"] == model))
        row = pd.concat([prev[~same], row], ignore_index=True)
    os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
    row[RESULT_COLS].to_csv(RESULTS_FILE, index=False)
    print(f"Saved: {model} ({activity}, {LANGUAGE})  RMSE = {float(rmse_value):.4f}")


def results_table():
    """Everything saved so far, by every block and language, best to worst."""
    if not os.path.exists(RESULTS_FILE) or os.path.getsize(RESULTS_FILE) == 0:
        print("No results saved yet.")
        return pd.DataFrame(columns=RESULT_COLS)
    t = pd.read_csv(RESULTS_FILE).sort_values("rmse_test").reset_index(drop=True)
    print(t.to_string())
    return t


def quarter_axis(ax):
    """One tick per test quarter, labelled by its quarter-end month (03, 06, 09, 12)."""
    ax.set_xticks(y_obs.index)
    ax.set_xticklabels([f"{d:%Y-%m}" for d in y_obs.index], rotation=45, ha="right")


def plot_observed_predicted(predicted, name, color="#8E44AD", ax=None):
    """Observed vs predicted GDP growth on the test sample."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 4))
    ax.plot(y_obs.index, y_obs.values, "o-", color="#2C3E50", lw=2, label="Observed")
    ax.plot(y_obs.index, np.asarray(predicted), "s--", color=color, lw=2, label=name)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_ylabel("Real GDP growth, q/q (%)")
    ax.set_title(f"Observed vs predicted: {name}")
    quarter_axis(ax)
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    return ax


benchmark_rmse = rmse(y_obs, np.repeat(y_train.mean(), len(y_obs)))
print(f"Train-mean benchmark on the test quarters: RMSE = {benchmark_rmse:.4f} "
      f"(predicting {y_train.mean():.2f}% every quarter)\n")
results_table();

## The data, the LSTM way

Look at the last rows of the file. GDP growth sits only on quarter-end months; the other two
months of each quarter have no target. Each training example is a window of `n_timesteps`
months of all 15 predictors ending on a quarter-end month, and its label is that quarter's GDP
growth (the sliding windows from the lecture).

**Standardizing.** As in Block 1, the predictors are put on a common scale using the train
window's mean and standard deviation. Neural networks train poorly when one input moves by 30
and another by 0.3. GDP growth, the target, stays in percent.

**Missing values.** `nowcast_lstm` fills gaps inside a series with `fill_na_func` and missing
months at the end (the ragged edge) with `fill_ragged_edges_func`. Both are the train-sample
mean here, which after standardizing is zero: "no news" for that series.

In [ ]:
train_mask = monthly["date"] <= train_end
mu = monthly.loc[train_mask, predictors].mean()
sd = monthly.loc[train_mask, predictors].std()

data_std = monthly.copy()
data_std[predictors] = (monthly[predictors] - mu) / sd

# Train window: every month up to the last train quarter
train_lstm = data_std[data_std["date"] <= train_end].reset_index(drop=True)
# History up to the last published GDP quarter, for scoring the test quarters
hist_lstm = data_std[data_std["date"] <= test_end].reset_index(drop=True)

print(f"Train window: {len(train_lstm)} months, "
      f"{train_lstm[target_variable].notna().sum()} of them with a GDP value")
data_std.tail(8).round(2)

### One training window, the object the network sees

Ridge, LASSO and the trees received one row per quarter. The LSTM receives a **block**: the
12 months that end on a quarter-end, all 15 series, and one answer, that quarter's GDP growth.
The cell below prints the block that ends in December 2019 (values in standard deviations). Read
it as the network does: top to bottom, month by month, remembering what it has seen so far, and
at the last row it has to say a number.

Count the blocks. The train window has 41 quarters with a GDP value, but the first three do not
have 12 months of history behind them, so the network has **38 training examples**. That number
is the key to everything that follows.

In [ ]:
N_TS = 12                                  # the pipeline's window: one year of months
window_end = pd.Timestamp("2019-12-01")
rows = train_lstm.index[train_lstm["date"] <= window_end][-N_TS:]
window = train_lstm.loc[rows, ["date"] + predictors].set_index("date")
window.columns = labels
window.index = [f"{d:%Y-%m}" for d in window.index]
label_value = float(train_lstm.loc[train_lstm["date"] == window_end, target_variable].iloc[0])

print(f"Input : {N_TS} months x {len(predictors)} series = {N_TS * len(predictors)} numbers")
print(f"Answer: GDP growth 2019Q4 = {label_value:.2f}%")

n_windows = int((train_lstm[target_variable].notna() & (train_lstm.index >= N_TS - 1)).sum())
print(f"Training examples with a full {N_TS}-month history: {n_windows}")
window.round(1)

## The knobs

The same table as in the lecture. The right-hand column is what this notebook uses. The only
change from the pipeline is `n_models`: 10 networks instead of 50, so the cells run in seconds
on a laptop. Everything else is the pipeline's setting.

| Parameter | Pipeline | Here | Intuition |
|---|---|---|---|
| `n_timesteps` | 12 | 12 (you will move it) | How many months back the LSTM looks. Longer = more context, harder optimization. |
| `n_models` | 50 | 10 | Bagging: train several LSTMs from different random starts and average them. |
| `train_episodes` | 100 | 100 (you will move it) | Full passes (epochs) over the training data. Too few = underfit; too many = overfit and drift. |
| `batch_size` | 50 | 50 | Training windows per gradient step. With 38 windows, every epoch is one step on all of them. |
| `decay` | 0.98 | 0.98 | Per-epoch learning-rate decay: the optimizer slows down as it nears a minimum. |
| `n_hidden` | 10 | 10 | Width of the hidden and cell state: the model's capacity. |
| `n_layers` | 1 | 1 | Stacked LSTM layers. One is enough for this little data. |
| `dropout` | 0.0 | 0.0 | We rely on bagging instead. |
| `criterion` | MSE | MSE | Average squared error. |
| `optimizer` (η) | Adam (0.01) | Adam (0.01) | Gradient-descent variant with initial learning rate η. |
| `seeds` | | 0, 1, ..., 9 | One per network, for reproducibility. |

**How many weights is that?** The lecture's four gates each carry a weight matrix on the input
(15 series) and one on the hidden state (10 units), plus biases:
4 × (10 × 15 + 10 × 10 + 2 × 10) = 1,080. The output layer reads the hidden state of all 12
months, 120 numbers, with one bias: 121. In total **1,201 weights to learn from 38 examples**,
about 32 per example. Keep that ratio in mind when you read the train error below.

In [ ]:
BASE_PARAMS = dict(
    n_timesteps=12,
    fill_na_func=np.nanmean,
    fill_ragged_edges_func=np.nanmean,
    n_models=10,
    train_episodes=100,
    batch_size=50,
    decay=0.98,
    n_hidden=10,
    n_layers=1,
    dropout=0.0,
    criterion=torch.nn.MSELoss(),
    optimizer=torch.optim.Adam,
    optimizer_parameters={"lr": 1e-2, "weight_decay": 0.0},
)


def fit_lstm(data=None, seeds=None, **changes):
    """Fit an LSTM on the train window (or `data`) with BASE_PARAMS, overridden by `changes`."""
    params = {**BASE_PARAMS, **changes}
    params["seeds"] = list(range(params["n_models"])) if seeds is None else list(seeds)
    model = LSTM(data=train_lstm if data is None else data, target_variable=target_variable, **params)
    model.device = "cpu"                                     # same numbers on every laptop
    model.train(quiet=True)
    return model


def score_lstm(model):
    """Train and test RMSE of a fitted LSTM, plus its test predictions."""
    p = model.predict(hist_lstm).set_index("date")["predictions"]
    pred_test = p.reindex(test_dates).values
    tr = p.reindex(train_dates).dropna()
    return rmse(y_train.loc[tr.index], tr), rmse(y_obs, pred_test), pred_test


def n_weights(model):
    """Number of trainable weights in one of the model's networks."""
    return sum(p.numel() for p in model.mv_lstm[0].parameters())

## A first LSTM, with the pipeline's settings

In [ ]:
t0 = time.time()
lstm_base = fit_lstm()
rmse_base_train, rmse_base, pred_base = score_lstm(lstm_base)

print(f"Fitted {lstm_base.n_models} networks in {time.time() - t0:.1f} s")
print(f"Training examples (windows x months x series): {np.asarray(lstm_base.X).shape}")
print(f"Weights per network: {n_weights(lstm_base):,}")
print(f"RMSE train : {rmse_base_train:.4f}")
print(f"RMSE test  : {rmse_base:.4f}   (train-mean benchmark: {benchmark_rmse:.4f})")
comparison_table(pred_base, "LSTM")

Look at the train RMSE: it is close to zero. With 1,201 weights and 38 examples, 100 epochs are
enough to learn the train quarters almost by heart, 2020 included. The test RMSE is what the
network can actually do, and it sits next to the benchmark. Also look at the sign of the
errors: most are negative. We come back to that in the 2020 exercise.

### Chart 1: The training loss, epoch by epoch

`nowcast_lstm` keeps the loss of every epoch in `.train_loss`, one curve per network. It falls
fast at first and then flattens, the left half of the lecture's picture. The library does not
track a validation loss, so to see the other half we refit with different numbers of epochs
and score each on the test quarters (next chart).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for k, loss in enumerate(lstm_base.train_loss):
    ax.plot(range(1, len(loss) + 1), loss, color="#8E44AD", lw=1, alpha=0.5,
            label="One network (seed)" if k == 0 else None)
ax.plot(range(1, len(loss) + 1), np.mean(lstm_base.train_loss, axis=0), color="#2C3E50",
        lw=2.5, label="Average of the 10")
ax.set_yscale("log")
ax.set_xlabel("Epoch (train_episodes)")
ax.set_ylabel("Training loss (MSE, log scale)")
ax.set_title("Training loss falls quickly, then flattens")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
plt.tight_layout()

### Exploring `train_episodes`: what 38 examples can teach

The same model with 5 to 200 epochs, scored on train and test. The lecture's picture has three
parts: the test error falls while the network learns, reaches a minimum, and rises when it
starts memorizing. The table adds a column the lecture's picture does not have: the standard
deviation of the 12 test predictions. When it is near zero the network is answering the same
number for every quarter.

In [ ]:
episodes = [5, 10, 25, 50, 100, 200]
rows = []
t0 = time.time()
for ep in episodes:
    m = fit_lstm(train_episodes=ep)
    r_tr, r_te, p_te = score_lstm(m)
    rows.append({"train_episodes": ep, "RMSE train": r_tr, "RMSE test": r_te,
                 "sd of test predictions": float(np.std(p_te))})
explore_ep = pd.DataFrame(rows)
print(f"{len(episodes)} configurations in {time.time() - t0:.0f} s")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(explore_ep["train_episodes"], explore_ep["RMSE train"], "o-", color="#2980B9", lw=2, label="Train")
ax.plot(explore_ep["train_episodes"], explore_ep["RMSE test"], "s-", color="#C0392B", lw=2, label="Test")
ax.axhline(benchmark_rmse, color="grey", ls=":", label="Train-mean benchmark (test)")
ax.set_xscale("log")
ax.set_xlabel("train_episodes (log scale)")
ax.set_ylabel("RMSE")
ax.set_title("LSTM: the train error falls, the test error does not follow")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
plt.tight_layout()

explore_ep.round(4)

Read the chart against the lecture's picture, part by part.

- **Left end.** After 5 epochs the network has barely moved from its random start: the sd of
  its predictions is close to zero, it answers the mean for every quarter, and its test RMSE
  lands on the benchmark. That is underfitting, and it can look deceptively good on a sample
  where nothing beats the mean by much.
- **Middle.** Between 10 and 50 epochs the train error collapses: the network is storing the 38
  examples. The test error does not fall with it. There is no epoch at which the network knows
  something about the test quarters that the mean did not already know.
- **Right end.** From 50 epochs on, both curves are flat: the examples are fully memorized and
  nothing is left to learn.

The lecture's picture, with a visible minimum, needs more examples than 38. Two lessons that do
survive on this sample: the train error of a network is not information about its accuracy,
and `train_episodes` is not the knob that makes this model better. What the network sees in
the window, and how many networks are averaged, matter more; that is where the next cells go.

### Exploring `n_timesteps`

How many months should the network look back? 12 is the pipeline's choice (a year). Shorter
windows give the network less to memorize and leave a few more training examples at the start
of the sample. The second cell refits the pipeline's setting with two other sets of seeds, to
show how much of a difference between two rows of the table is just the random start.

In [ ]:
timesteps = [3, 6, 9, 12]
rows = []
t0 = time.time()
for nts in timesteps:
    m = fit_lstm(n_timesteps=nts)
    r_tr, r_te, _ = score_lstm(m)
    rows.append({"n_timesteps": nts, "training examples": len(m.y), "weights": n_weights(m),
                 "RMSE train": r_tr, "RMSE test": r_te})
explore_ts = pd.DataFrame(rows)
print(f"{len(timesteps)} configurations in {time.time() - t0:.0f} s")
explore_ts.round(4)

In [ ]:
# The seed band: the same model (12 months, 100 epochs), other random starts
band = [rmse_base]
for first in (10, 20):
    band.append(score_lstm(fit_lstm(seeds=range(first, first + 10)))[1])
print("Test RMSE of the pipeline's setting with seeds 0-9, 10-19, 20-29:",
      ", ".join(f"{b:.3f}" for b in band))
print(f"Spread: {max(band) - min(band):.3f} pp. A difference between two rows of the table "
      f"smaller than this is noise from the random start.")

### YOUR TURN: fix the final model

From the two tables, pick `n_timesteps` and `train_episodes` and set them here. Compare the
differences between rows with the seed band you just printed before you call one setting
better than another. This LSTM is saved as the block's official result. (Same caution as in
Blocks 1 and 2: choosing on the test RMSE is fine for learning; a reported model would be
tuned on the train window only.)

In [ ]:
# ============ YOUR TURN ============
# Copy here the values you chose from the two exploration tables above.
n_timesteps = 12
train_episodes = 100
# ===================================

t0 = time.time()
lstm = fit_lstm(n_timesteps=n_timesteps, train_episodes=train_episodes)
rmse_lstm_train, rmse_lstm, pred_lstm = score_lstm(lstm)

print(f"Fitted in {time.time() - t0:.1f} s")
print(f"RMSE train : {rmse_lstm_train:.4f}")
print(f"RMSE test  : {rmse_lstm:.4f}   (pipeline settings: {rmse_base:.4f}, benchmark: {benchmark_rmse:.4f})")

plot_observed_predicted(pred_lstm, "LSTM")
save_result(3, "act8", "LSTM", rmse_lstm,
            f"n_timesteps={n_timesteps}, train_episodes={train_episodes}, n_models={lstm.n_models}, seeds 0-{lstm.n_models - 1}")

### Chart 2: Bagging: ten networks, one forecast

`n_models` plays the role of the trees in a random forest. Each network starts from different
random weights and ends somewhere different; the LSTM's forecast is their average. The grey
lines are the ten networks one by one on the test quarters, the purple line is the average.
The right panel shows each network's RMSE against the average's.

In [ ]:
per_net = []
for net in lstm.mv_lstm:
    one = copy.copy(lstm)                 # same data and settings ...
    one.mv_lstm, one.n_models = [net], 1  # ... but a single network
    p = one.predict(hist_lstm).set_index("date")["predictions"]
    per_net.append(p.reindex(test_dates).values)
per_net = np.array(per_net)
rmse_nets = np.array([rmse(y_obs, p) for p in per_net])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1.6, 1]})
for p in per_net:
    ax1.plot(y_obs.index, p, color="grey", lw=0.8, alpha=0.6)
ax1.plot(y_obs.index, y_obs.values, "o-", color="#2C3E50", lw=2.5, label="Observed")
ax1.plot(y_obs.index, pred_lstm, "s-", color="#8E44AD", lw=2.5, label="Average of the networks")
ax1.axhline(0, color="grey", lw=0.8)
ax1.set_ylabel("Real GDP growth, q/q (%)")
ax1.set_title("Each grey line is one network (seed)")
quarter_axis(ax1)
ax1.legend(frameon=False)
ax1.grid(alpha=0.25)

ax2.bar(range(len(rmse_nets)), np.sort(rmse_nets), color="lightgrey", edgecolor="grey")
ax2.axhline(rmse_lstm, color="#8E44AD", lw=2.5, label=f"Average ({rmse_lstm:.2f})")
ax2.axhline(benchmark_rmse, color="grey", ls=":", label=f"Benchmark ({benchmark_rmse:.2f})")
ax2.set_xlabel("Networks, sorted by RMSE")
ax2.set_ylabel("Test RMSE")
ax2.set_title("Error of each network")
ax2.legend(frameon=False)
plt.tight_layout()

print(f"The average beats {(rmse_nets > rmse_lstm).mean():.0%} of its own networks")

## One quarter can set the level: the 2020 exercise

Go back to the comparison table of the first LSTM: most errors have the same sign. The network
is too low in nearly every test quarter. The first cell below separates the error into two
parts: the **mean error** (how far the level is off, on average) and what is left after
removing it (how well the ups and downs are tracked).

Where does a level error come from in a model trained on 38 examples? One of them is 2020Q2,
with GDP growth of −12%. The loss is the squared error, so that one example weighs as much as
dozens of ordinary quarters, and the network bends its level toward it. The second cell
removes the four 2020 quarters from the **answers** (the predictors stay, so the windows are
intact), refits with the same settings, and compares. This is a measurement, not a
recommendation: the nowcast tool keeps 2020 in the sample and treats quarters like it as a
separate regime. The point here is to see how much weight one example carries.

In [ ]:
err = pred_lstm - y_obs.values
print(f"Test RMSE            : {rmse_lstm:.4f}")
print(f"Mean error (level)   : {err.mean():+.3f} pp   <- negative: the network sits below the actuals")
print(f"RMSE without the level error: {np.sqrt(np.mean((err - err.mean()) ** 2)):.4f}")
print(f"Correlation of predictions with actuals: {np.corrcoef(pred_lstm, y_obs.values)[0, 1]:+.2f}")

In [ ]:
# The same model, with the 2020 answers set to missing (34 examples instead of 38)
train_no2020 = train_lstm.copy()
train_no2020.loc[train_no2020["date"].dt.year == 2020, target_variable] = np.nan

t0 = time.time()
lstm_no2020 = fit_lstm(data=train_no2020, n_timesteps=n_timesteps, train_episodes=train_episodes)
_, rmse_no2020, pred_no2020 = score_lstm(lstm_no2020)
err_no2020 = pred_no2020 - y_obs.values
print(f"Fitted in {time.time() - t0:.1f} s on {len(lstm_no2020.y)} examples")

pd.DataFrame({
    "training examples": [len(lstm.y), len(lstm_no2020.y)],
    "RMSE test": [rmse_lstm, rmse_no2020],
    "mean error (pp)": [err.mean(), err_no2020.mean()],
    "RMSE without the level error": [np.sqrt(np.mean((err - err.mean()) ** 2)),
                                     np.sqrt(np.mean((err_no2020 - err_no2020.mean()) ** 2))],
}, index=["with 2020", "2020 answers removed"]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(y_obs.index, y_obs.values, "o-", color="#2C3E50", lw=2.5, label="Observed")
ax.plot(y_obs.index, pred_lstm, "s--", color="#8E44AD", lw=2, label=f"LSTM, 2020 in the answers ({rmse_lstm:.2f})")
ax.plot(y_obs.index, pred_no2020, "d--", color="#E67E22", lw=2, label=f"LSTM, 2020 answers removed ({rmse_no2020:.2f})")
ax.axhline(y_train.mean(), color="grey", ls=":", lw=1.5, label="Train mean")
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("Real GDP growth, q/q (%)")
ax.set_title("One example out of 38 moves the whole level")
quarter_axis(ax)
ax.legend(frameon=False, ncol=2)
ax.grid(alpha=0.25)
plt.tight_layout()

## The ragged edge: the reason to use an LSTM for nowcasting

So far every test quarter was predicted with all three of its months available. In real life
the nowcast is made while the quarter is still under way, and each series arrives with its own
delay. The current end of the file shows the pattern: count, for each series, how many months
behind the latest month (October 2026) its last observation is. That is its publication delay.

In [ ]:
last_month = monthly["date"].max()
last_obs = monthly.set_index("date")[predictors].apply(lambda s: s.last_valid_index())
pub_lags = [(last_month.year - d.year) * 12 + (last_month.month - d.month) for d in last_obs]

lags_table = pd.DataFrame({"label": labels, "last observation": last_obs.dt.strftime("%Y-%m").values,
                           "publication delay (months)": pub_lags}, index=predictors)
lags_table.sort_values("publication delay (months)")

### One quarter as it looked while it was under way

Take the last test quarter, 2026Q1, and rebuild its window at three moments: the first month
of the quarter (`lag = -2`), its last month (`lag = 0`), and two months after it ended
(`lag = +2`, about when GDP is published). At each moment a series is missing its last
`publication delay − lag` months of the window, and `nowcast_lstm` fills them with "no news"
(the train mean, zero after standardizing). The first table counts, for each series, how many
of the window's 12 months the network actually sees at each moment. The second gives the
prediction at each moment, next to the prediction with the full window and the actual. For this
quarter the three numbers barely move: the series that carry the signal (energy, oil, lights)
are the ones with no delay, so most of the window is already there on the first month.

This is the operation none of the other models in the workshop can do on their own: predict a
quarter from a window with holes at the end, without re-estimating anything.

In [ ]:
quarter = pd.Timestamp("2026-03-01")
moments = [-2, 0, 2]

seen = pd.DataFrame(index=labels)
seen["publication delay"] = pub_lags
for lag in moments:
    seen[f"months seen at lag {lag:+d}"] = [max(0, n_timesteps - max(0, pl - lag)) for pl in pub_lags]
print(f"Window of {n_timesteps} months ending {quarter:%Y-%m}: months the network sees, by series")
display(seen.sort_values("publication delay"))

one = {"actual": float(y_obs.loc[quarter]), "full window": float(pred_lstm[list(test_dates).index(quarter)])}
for lag in moments:
    rp = lstm.ragged_preds(pub_lags, lag, data=data_std,
                           start_date=f"{quarter:%Y-%m-%d}", end_date=f"{quarter:%Y-%m-%d}")
    one[f"lag {lag:+d}"] = float(rp["predictions"].iloc[0])
pd.Series(one, name=f"GDP growth {quarter.year}Q{quarter.quarter} (%)").round(2).to_frame()

### Chart 3: How good is the nowcast at each point of the quarter?

The same operation for all 12 test quarters at five moments, `lag = -2` to `+2`. These are
the vintages p = −2 … +2 of the nowcast tool. The blanked months are filled with "no news", so
early in the quarter the LSTM leans on the older months it does see, and the error falls as the
vintage moves right. Read the chart against the benchmark line too: on these 12 quarters the
LSTM does not get below the train mean at any vintage. What the model buys is not accuracy on
this sample but the ability to update the number month by month as the series arrive, which a
constant, a regression on quarterly averages, or a tree cannot do without being rebuilt.

In [ ]:
vintages = [-2, -1, 0, 1, 2]
vint_rmse, vint_preds = [], {}
for lag in vintages:
    rp = lstm.ragged_preds(pub_lags, lag, data=data_std,
                           start_date=f"{test_dates.min():%Y-%m-%d}", end_date=f"{test_end:%Y-%m-%d}")
    rp = rp.set_index("date").reindex(test_dates)
    vint_preds[lag] = rp["predictions"].values
    vint_rmse.append(rmse(y_obs, rp["predictions"]))

vint = pd.DataFrame({"vintage (lag, months)": vintages, "RMSE test": vint_rmse})

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(vintages, vint_rmse, "o-", color="#8E44AD", lw=2, label="LSTM, simulated vintages")
ax.axhline(rmse_lstm, color="#8E44AD", ls="--", lw=1.2, label="LSTM, all data available")
ax.axhline(benchmark_rmse, color="grey", ls=":", label="Train-mean benchmark")
ax.set_xticks(vintages)
ax.set_xlabel("Months relative to the end of the quarter")
ax.set_ylabel("Test RMSE")
ax.set_title("Nowcast error as the quarter's data arrive")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
plt.tight_layout()

vint.round(4)

### The nowcast for 2026Q2 and 2026Q3, with today's data

GDP for 2026Q2 is not published yet, and 2026Q3 ended a month ago. Neither has a GDP value in
the file, but `predict()` still returns a prediction for every month that has a full window of
predictors behind it. It fills each series' missing months with "no news" and predicts. The
table shows which predictor months exist for each quarter; the more of them are missing, the
more the nowcast falls back on older information.

In [ ]:
live = lstm.predict(data_std).set_index("date")["predictions"]
nowcast = pd.DataFrame({
    "quarter": ["2026Q2", "2026Q3"],
    "LSTM nowcast (q/q %)": [live.loc["2026-06-01"], live.loc["2026-09-01"]],
}).set_index("quarter").round(2)

# Which months of each quarter are available, per series ("x" = observed, "." = missing)
months = pd.date_range("2026-04-01", "2026-09-01", freq="MS")
avail = (monthly.set_index("date").loc[months, predictors].notna()
         .T.replace({True: "x", False: "."}))
avail.columns = [f"{d:%Y-%m}" for d in months]
avail.index = labels
avail["months available"] = (monthly.set_index("date").loc[months, predictors].notna().sum()).values

print(nowcast.to_string())
print(f"\nLast published quarter, 2026Q1: {y_obs.iloc[-1]:.2f}%   "
      f"train-sample mean: {y_train.mean():.2f}%")
avail

## Wrap-up: the full table

Every model of the three blocks, in Python and in R, scored on the same 12 test quarters. The
chart puts them side by side with the train-mean benchmark.

Questions for the group:

- How many models beat the train mean, and by how much? On 12 quarters, a few tenths of a
  point is within what a different test window could reverse, and within the LSTM's seed band.
- The LSTM is the only model that reads the months inside the quarter and handles the ragged
  edge on its own. Even when its RMSE on complete quarters is not the best, that is why the
  nowcast tool keeps it in the mix.
- What did 38 examples teach the network, and what did they not? Which of the three levers you
  touched (epochs, window length, number of networks) changed the test error by more than the
  seed band?

In [ ]:
res = results_table()
if len(res):
    res = res.sort_values("rmse_test", ascending=True)
    names = res["model"] + " (" + res["language"] + ")"
    block_colors = {0: "#95A5A6", 1: "#2980B9", 2: "#27AE60", 3: "#8E44AD"}
    fig, ax = plt.subplots(figsize=(10, 0.4 * len(res) + 1.5))
    ax.barh(names, res["rmse_test"], color=[block_colors.get(b, "#E67E22") for b in res["block"]])
    ax.axvline(benchmark_rmse, color="black", ls=":", lw=1.5, label="Train-mean benchmark")
    ax.invert_yaxis()
    ax.set_xlabel("Test RMSE (pp of q/q GDP growth)")
    from matplotlib.patches import Patch
    names_b = {0: "Benchmarks", 1: "Block 1: regularization", 2: "Block 2: trees", 3: "Block 3: LSTM"}
    handles = [Patch(color=block_colors.get(b, "#E67E22"), label=names_b.get(b, f"Block {b}"))
               for b in sorted(res["block"].unique())]
    handles += ax.get_legend_handles_labels()[0]
    ax.legend(handles=handles, frameon=True, loc="upper right")
    ax.set_title("Workshop results on the 12 test quarters")
    ax.grid(alpha=0.25, axis="x")
    plt.tight_layout()